<a href="https://colab.research.google.com/github/BrendoAires/ChurnFinanceiro/blob/main/Churn_Financeiro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Previsão de Churn de Clientes utilizando Redes Neurais com Keras

#####-Este projeto desenvolve um modelo de aprendizado profundo focado na previsão de **Churn**  com base no histórico de uma instituição financeira.
#####-O objetivo principal é identificar quais clientes apresentam alta probabilidade de cancelar seus serviços, permitindo que a equipe de negócios atue preventivamente com estratégias de retenção.

#####-Os dados utilizados contêm características demográficas e comportamentais dos clientes, como idade, saldo bancário, quantidade de produtos contratados e se o cliente é ativo.

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from keras.models import Sequential
from keras.layers import Dense, Dropout

In [7]:
df = pd.read_csv("Churn_treino.csv", sep=";")
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0,1,1,1,10134888,1
1,608,Spain,Female,41,1,8380786,1,0,1,11254258,0
2,502,France,Female,42,8,1596608,3,1,0,11393157,1
3,699,France,Female,39,1,0,2,0,0,9382663,0
4,850,Spain,Female,43,2,12551082,1,1,1,790841,0


## 1. Pré-processamento e Engenharia de Recursos (Feature Engineering)

Para que a rede neural possa processar e extrair o padrão dos dados, realizamos duas etapas fundamentais de transformação:

* **Padronização das Variáveis Numéricas (`StandardScaler`):** Redes neurais utilizam algoritmos baseados em descida de gradiente, que convergem muito mais rápido se os dados numéricos estiverem na mesma escala (média 0 e desvio padrão 1).
* **Codificação de Variáveis Categoriáveis (`LabelEncoder`):** Modelos matemáticos não processam textos diretamente. Utilizamos o codificador para transformar colunas de texto (como gênero ou país) em representações numéricas sequenciais.

In [8]:
X = df.drop("Exited", axis=1)
y = df["Exited"]

In [12]:
standardscaler = StandardScaler()
numerical = X.select_dtypes(include=['int64', 'float64']).columns
X[numerical] = standardscaler.fit_transform(X[numerical])

In [15]:
labelencoder = LabelEncoder()
categorical = X.select_dtypes(include='object').columns
for col in categorical:
  X[col] = labelencoder.fit_transform(X[col])


In [17]:
X

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,-0.326221,0,0,0.293517,-1.041760,-1.110553,-0.911583,0.646092,0.970243,0.170614
1,-0.440036,2,0,0.198164,-1.387538,0.222782,-0.911583,-1.547768,0.970243,0.353281
2,-1.536794,0,0,0.293517,1.032908,-0.856542,2.527057,0.646092,-1.030670,0.375948
3,0.501521,0,0,0.007457,-1.387538,-1.110553,0.807737,-1.547768,-1.030670,0.047859
4,2.063884,2,0,0.388871,-1.041760,0.886252,-0.911583,0.646092,0.970243,-1.354223
...,...,...,...,...,...,...,...,...,...,...
9995,1.246488,0,1,0.007457,-0.004426,-1.110553,0.807737,0.646092,-1.030670,0.087743
9996,-1.391939,0,1,-0.373958,1.724464,-0.197835,-0.911583,0.646092,0.970243,0.176340
9997,0.604988,0,0,-0.278604,0.687130,-1.110553,-0.911583,-1.547768,0.970243,-0.796492
9998,1.256835,1,1,0.293517,-0.695982,0.083852,0.807737,0.646092,-1.030670,0.032551


## 2. Divisão dos Dados e Modelo (Keras)

A base de dados foi dividida utilizando a proporção de **70% para treinamento** e **30% para teste**, garantindo que o modelo seja avaliado em dados que ele nunca viu antes.

A rede neural foi construída de forma sequencial com a seguinte lógica de arquitetura:
* **Camadas Ocultas (`Dense`):** Utilizamos camadas densamente conectadas com a função de ativação `ReLU` para introduzir não-linearidade no aprendizado.
* **Prevenção de Overfitting (`Dropout`):** Adicionamos camadas de Dropout configuradas em 40% (`0.4`) para desativar neurônios aleatoriamente a cada época de treino, forçando a rede a criar representações mais robustas e generalistas.
* **Camada de Saída:** Utiliza um único neurônio com a função de ativação `Sigmoid`, ideal para problemas de **classificação binária**, pois mapeia a saída em uma probabilidade entre 0 e 1 (onde > 0.5 indica propensão ao Churn).

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state =0)

In [36]:
model = Sequential()
model.add(Dense(units=64, activation='relu', input_dim=X_train.shape[1]))
model.add(Dropout(0.4))
model.add(Dense(units=32, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(units=64, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(units=1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [38]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=100, batch_size=32)

Epoch 1/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8544 - loss: 0.3573
Epoch 2/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8586 - loss: 0.3516
Epoch 3/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8613 - loss: 0.3523
Epoch 4/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8566 - loss: 0.3516
Epoch 5/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8581 - loss: 0.3489
Epoch 6/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8589 - loss: 0.3549
Epoch 7/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8569 - loss: 0.3523
Epoch 8/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8593 - loss: 0.3519
Epoch 9/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8586 - loss: 0.3521
Epoch 10/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8567 - loss: 0.3457
Epoch 11/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8561 - loss: 0.3516
Epoch 12/100
219/219 ━━━━━━━━━━━━━━━━━━━━

In [39]:
previsao = model.predict(X_test)
previsao

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


array([[0.2291766 ],
       [0.25815615],
       [0.1839191 ],
       ...,
       [0.08177906],
       [0.13829023],
       [0.59176195]], dtype=float32)

In [40]:
y_pred = (previsao>0.5).astype('int32')
y_pred

array([[0],
       [0],
       [0],
       ...,
       [0],
       [0],
       [1]], dtype=int32)

## 3. Avaliação de Desempenho do Modelo

Após o treinamento ao longo de 100 épocas, avaliamos a eficácia do classificador na base de testes através de diferentes métricas:

### Resultados Obtidos:
* **Acurácia Geral:** **86.16%** (O modelo acerta a maior parte das previsões no geral).
* **F1-Score:** **59.11%** (Média harmônica que equilibra a precisão e a sensibilidade).
* **Recall (Sensibilidade):** **48.31%** (A capacidade real do modelo em encontrar os clientes que de fato vão sair da empresa).

### Matriz de Confusão:
```text
[[2285   94]  -> [Verdadeiros Negativos, Falsos Positivos]
 [ 321  300]] -> [Falsos Negativos, Verdadeiros Positivos]


In [41]:
print('Acurácia: ', accuracy_score(y_test, y_pred))
print('F1 Score: ', f1_score(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred))
print('Matriz de confusão: \n', confusion_matrix(y_test, y_pred))

Acurácia:  0.8616666666666667
F1 Score:  0.5911330049261084
Recall:  0.4830917874396135
Matriz de confusão: 
 [[2285   94]
 [ 321  300]]
